# Mitra v2

**Estado: nunca executado.** Este notebook precisa de GPU e para na primeira célula se não houver uma.
O código foi conferido contra a fonte do `autogluon.tabular` 1.6.3 e testado com um classificador falso em `tests/test_mitra.py`, mas o modelo real ainda não gerou nenhuma previsão.
Nenhum número abaixo existe até a primeira execução. O passo a passo está na seção "Mitra v2" do `README.md`.

Modelo, faixas e trial 0 seguem a Frente 5 de `planos/rafael.md`.
O conjunto de teste não é usado neste notebook.

A variável `MITRA_ETAPA` escolhe o que roda:

| Etapa | O que faz |
| --- | --- |
| `medidas` (padrão) | As três medidas do plano: `bfloat16`, reprodutibilidade e tempo dos trials |
| `curto` | Trial 0 e mais 4 trials em `resultados/curto/`, banco descartável e fora do Git |
| `completo` | Estudo em `resultados/mitra.db`, com o N em `MITRA_N_TRIALS` |

In [ ]:
import sys

sys.path.insert(0, "..")

import gc
import os
import platform
import time

import numpy as np
import optuna
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Mitra só roda com GPU neste projeto. Veja a seção Mitra v2 do README.")

import autogluon.tabular
from autogluon.tabular.models.mitra.sklearn_interface import MitraClassifier

import comum

print(sys.executable)
print("Python", platform.python_version(), "| torch", torch.__version__, "| autogluon.tabular", autogluon.tabular.__version__)
memoria_gpu = torch.cuda.get_device_properties(0).total_memory / 2**30
print("GPU:", torch.cuda.get_device_name(0), f"{memoria_gpu:.1f} GB")

In [ ]:
MODELO = "mitra"
HF_MODEL = "autogluon/mitra-classifier-2"
ETAPA = os.environ.get("MITRA_ETAPA", "medidas")
if ETAPA not in ("medidas", "curto", "completo"):
    raise ValueError(f"MITRA_ETAPA desconhecida: {ETAPA}")
ESTUDO_CURTO = ETAPA != "completo"
N_TRIALS = int(os.environ["MITRA_N_TRIALS"]) if ETAPA == "completo" else 5  # N final ainda não definido
CONTEXTO_PADRAO = 8192  # max_samples_support do AutoGluon, acima das 6.208 linhas da dobra

# O teste, que vem em `final`, fica de fora de propósito.
folds_por_oversampling = {
    True: comum.carregar_dobras()[0],
    False: comum.carregar_dobras(balancear=False)[0],
}
print({k: len(v[0][0]) for k, v in folds_por_oversampling.items()}, "linhas de contexto na dobra 0")

## Decisões vindas da leitura do código

- **Sementes:** `seed` só controla a amostragem do contexto. O espelho de colunas e o embaralhamento vêm de `np.random`, e os pesos do fine-tuning, do torch; por isso `semear()` fixa os dois antes de cada `fit`. Se isso basta é a medida 2.
- **Falta de memória na GPU:** o `fit` captura `OutOfMemoryError`, corta o contexto pela metade e continua, só com um `print`. Como isso mudaria o modelo sem aviso, `ajustar` levanta erro se o contexto final não for o padrão.
- **Perda de treino:** `avaliar_cv` prevê o treino sem cópias com esse mesmo treino como contexto, então cada cliente aparece no próprio contexto com o rótulo certo. Essa perda não mede generalização e não deve ser lida como nos modelos treinados.
- **Custo da perda de treino:** essa previsão pode dominar o tempo do trial. A medida 3 separa o tempo de `fit`, de validação e de treino.

In [ ]:
FIXOS = dict(warmup_steps=10, patience=10, use_random_transforms=False, metric="log_loss")

PADRAO = dict(
    n_estimators=1,
    random_mirror_x=True,
    shuffle_features=False,
    shuffle_classes=False,
    oversampling_contexto=True,
    fine_tune=False,
)


def sugerir(t):
    hp = dict(
        n_estimators=t.suggest_categorical("n_estimators", [1, 2, 4, 8]),
        random_mirror_x=t.suggest_categorical("random_mirror_x", [True, False]),
        shuffle_features=t.suggest_categorical("shuffle_features", [True, False]),
        shuffle_classes=t.suggest_categorical("shuffle_classes", [True, False]),
        oversampling_contexto=t.suggest_categorical("oversampling_contexto", [True, False]),
        fine_tune=t.suggest_categorical("fine_tune", [True, False]),
    )
    if hp["fine_tune"]:
        hp["lr"] = t.suggest_float("lr", 1e-6, 1e-4, log=True)
        hp["fine_tune_steps"] = t.suggest_int("fine_tune_steps", 10, 100, step=10)
    return hp


def semear():
    np.random.seed(comum.SEMENTE)
    torch.manual_seed(comum.SEMENTE)
    torch.cuda.manual_seed_all(comum.SEMENTE)


def liberar_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def ajustador(hp):
    """Devolve o `ajustar` de comum.avaliar_cv."""
    params = {k: v for k, v in hp.items() if k != "oversampling_contexto"}

    def ajustar(dobra):
        Xt, yt, Xv, yv = dobra
        liberar_gpu()
        semear()
        modelo = MitraClassifier(hf_model=HF_MODEL, device="cuda", seed=comum.SEMENTE, verbose=False, **FIXOS, **params)
        modelo.fit(Xt, yt.astype(int), Xv, yv.astype(int))  # rótulos inteiros, por precaução
        contexto = modelo.trainers[0].cfg.hyperparams["max_samples_support"]
        if contexto != CONTEXTO_PADRAO:
            raise RuntimeError(f"Sem memória na GPU: o AutoGluon reduziu o contexto para {contexto} linhas")
        return lambda X: modelo.predict_proba(X)[:, 1]

    return ajustar


def objetivo(trial):
    hp = sugerir(trial)
    return comum.avaliar_cv(ajustador(hp), folds_por_oversampling[hp["oversampling_contexto"]], trial)["ks"]

## Medidas do plano

Só rodam com `MITRA_ETAPA=medidas`.

### 1. O `autocast` em `bfloat16` funciona nesta GPU?

In [ ]:
if ETAPA == "medidas":
    print("bfloat16 aceito (com emulação):", torch.cuda.is_bf16_supported())
    print("bfloat16 nativo:", torch.cuda.is_bf16_supported(including_emulation=False))
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        saida = torch.ones(8, 8, device="cuda") @ torch.ones(8, 8, device="cuda")
    print("tipo da saída no autocast:", saida.dtype)

### 2. A mesma semente repete as probabilidades?

Isso decide se o Mitra roda com 5 sementes no teste. Usa a dobra 0, duas vezes.

In [ ]:
if ETAPA == "medidas":
    dobra0 = folds_por_oversampling[True][0]
    previsoes = []
    for rodada in range(2):
        t0 = time.perf_counter()
        previsoes.append(ajustador(PADRAO)(dobra0)(dobra0[2]))
        ks_rodada = comum.ks(dobra0[3], previsoes[-1])
        print(f"rodada {rodada}: KS {ks_rodada:.6f} em {time.perf_counter() - t0:.1f} s")
    print("maior diferença absoluta entre as rodadas:", float(np.max(np.abs(previsoes[0] - previsoes[1]))))
    print("idênticas:", bool(np.array_equal(previsoes[0], previsoes[1])))

### 3. Quanto custa um trial?

Os dois trials do plano para fixar o N. O caro fica por último porque é o que mais pode faltar memória.

In [ ]:
def medir_trial(nome, hp):
    """Trial completo de 5 dobras, com o tempo separado por fase e o pico de memória da GPU."""
    tempos = dict(fit=0.0, validacao=0.0, treino=0.0)
    ajustar_base = ajustador(hp)

    def ajustar(dobra):
        t = time.perf_counter()
        prever_base = ajustar_base(dobra)
        tempos["fit"] += time.perf_counter() - t

        def prever(X):
            t = time.perf_counter()
            p = prever_base(X)
            tempos["validacao" if len(X) == len(dobra[2]) else "treino"] += time.perf_counter() - t
            return p

        return prever

    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    r = comum.avaliar_cv(ajustar, folds_por_oversampling[hp["oversampling_contexto"]])
    total = time.perf_counter() - t0
    print(nome)
    print(f"  KS {r['ks']:.4f} | por dobra {np.round(r['ks_dobras'], 4)} | limiar {r['limiar']:.4f}")
    print(f"  {total:.0f} s no trial | " + " | ".join(f"{fase} {s:.0f} s" for fase, s in tempos.items()))
    print(f"  pico de memória na GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
    print(f"  se todos os trials custassem isso: 100 trials levariam {total * 100 / 3600:.1f} h")
    return r

In [ ]:
if ETAPA == "medidas":
    r_contexto = medir_trial("Trial em contexto (trial 0)", PADRAO)

In [ ]:
if ETAPA == "medidas":
    caro = dict(PADRAO, n_estimators=8, fine_tune=True, lr=1e-4, fine_tune_steps=100)
    r_caro = medir_trial("Fine-tuning de 100 passos com n_estimators=8", caro)

## Estudo

Só roda com `MITRA_ETAPA=curto` ou `completo`.
Com `n_startup_trials=5`, o `MedianPruner` não corta nada nos 5 primeiros trials.

In [ ]:
if ETAPA != "medidas":
    pasta = comum.PASTA_RESULTADOS / "curto" if ESTUDO_CURTO else comum.PASTA_RESULTADOS
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / f"{MODELO}.db"
    if ESTUDO_CURTO:
        arquivo.unlink(missing_ok=True)
    armazenamento = f"sqlite:///{arquivo.as_posix()}"

    # Mesma configuração de estudo() em modelos_jeff.py, até o criar_estudo do Diego entrar no comum.py.
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    estudo = optuna.create_study(
        study_name=MODELO,
        storage=armazenamento,
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=comum.SEMENTE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    if not estudo.trials:
        estudo.enqueue_trial(PADRAO)  # trial 0 = padrão

    restantes = N_TRIALS - sum(t.state.is_finished() for t in estudo.trials)
    t0 = time.perf_counter()
    if restantes > 0:
        estudo.optimize(objetivo, n_trials=restantes, gc_after_trial=True)
    tempo_parede = estudo.user_attrs.get("tempo_parede_s", 0) + time.perf_counter() - t0
    estudo.set_user_attr("tempo_parede_s", tempo_parede)
    print(f"{restantes} trials em {tempo_parede:.1f} s")

In [ ]:
if ETAPA != "medidas":
    tabela = estudo.trials_dataframe(attrs=("number", "value", "state", "params"))
    tabela.columns = [c.removeprefix("params_") for c in tabela.columns]
    print(tabela.round(4).T.to_string(header=False))

    relido = optuna.load_study(study_name=MODELO, storage=armazenamento)
    assert [t.value for t in relido.trials] == [t.value for t in estudo.trials]  # o SQLite guardou o estudo
    assert relido.trials[0].params == PADRAO

    melhor = estudo.best_trial
    print("melhor trial:", melhor.number, "KS", round(melhor.value, 4))
    print("KS por dobra:", np.round(melhor.user_attrs["ks_dobras"], 4))
    for campo, valor in comum.resumir_estudo(estudo, N_TRIALS, tempo_parede).items():
        print(campo, "=", valor)

## Próxima etapa

Depende do N e usa o teste, então ainda não há código:

1. Treino final por retreino, com o contexto do treino inteiro: 7.760 linhas com oversampling ou 5.282 sem, conforme o melhor trial.
2. Se o fine-tuning vencer, o `fit` recebe o próprio treino sem cópias como `X_val`, e isso fica registrado como limitação.
3. Número de sementes conforme a medida de reprodutibilidade.
4. Gravação de `resultados/mitra.json` com `comum.salvar_resultado`.

---

# Roteiro original (21/10)

Mantido como registro.
A pré-proposta substituiu o protocolo de blocos pela busca conjunta no Optuna, então as seções abaixo não descrevem o que este notebook faz.

## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

O checkpoint é o v2, `autogluon/mitra-classifier-2`. Sem `hf_model`, o AutoGluon carrega o v1. A arquitetura dos dois é a mesma e está congelada: 12 camadas, largura 512, 4 cabeças, e o feed-forward interno com dimensão 2.048. Cerca de 20 colunas cabem nas 50 features do pré-treino v2 e passam das 16 do v1. Cerca de 7 mil linhas passam do suporte de pré-treino dos dois, 512 no v1 e 5.120 no v2.

Nenhuma fonte diz se a inferência cabe numa GPU de 6 GB. A heurística de reserva do AutoGluon, para este tamanho, dá cerca de 15,5 GB. O arquivo de pesos tem 303 MB, e isso não é o pico de memória. Se faltar memória, o código reduz o suporte.

O default do código é `fine_tune=True`, com 50 passos. O bloco 0 desliga isso.

No modo em contexto, um bloco cada:

- `n_estimators`. Default 1. O bloco sobe o número de cópias cujas probabilidades são promediadas. O artigo usa 32 no ensemble de contexto puro; 32 só entra se os valores menores tiverem vencido, porque trinta e duas cópias pesam na GPU.
- `shuffle_features`. Default `False`. Permuta a ordem das colunas.
- `shuffle_classes`. Default `False`. Permuta a ordem das classes e desfaz na saída.
- `use_random_transforms`. Default `False`. Liga o transformador aleatório do pré-processador, quantil com SVD ou passthrough.
- `random_mirror_x`. Default `True`. Multiplica cada coluna por +1 ou −1. O bloco que desliga só fica se vencer esse default.

O fine-tuning só abre se vencer o contexto. Não há argumento de batch, dropout, weight decay nem de otimizador no dicionário `MITRA`. O otimizador gravado é AdamW. A ativação gravada é GELU. O weight decay de estoque é 0,1. A receita publicada do v2 usa 0,3, e esse número não passa pelo dicionário.

- `fine_tune=True`, sozinho, testa o estoque: 50 passos, `lr` 0,0001, `warmup_steps` 1.000 e `patience` 40. Com esses defaults os 50 passos cabem dentro da rampa, e a taxa nominal quase não chega a valer.
- `warmup_steps`. A receita v2 usa 10. Com a taxa ainda em 0,0001, depois do décimo passo a taxa fica no valor nominal.
- `lr`. A receita v2, em classificação binária com até 16.384 linhas de treino, usa 3e-6. Esse bloco vem depois do warmup curto; com a rampa de 1.000 ainda ligada, essa taxa quase não é alcançada nos 50 passos.
- `patience`. Default 40. Só varia se o early stopping estiver cortando os 50 passos. O checkpoint guarda o menor log loss. Acurácia e AUC não entram aqui: o código trata o número como “menor é melhor”.

`fine_tune_steps` permanece 50. É o default, o valor do artigo e o da receita v2.

O oversampling repetitivo das dobras vale nos dois modos.

Fontes: arXiv:2510.21204, o relatório v2 arXiv:2609.04540, os cards `autogluon/mitra-classifier-2` e `autogluon/mitra-finetune`, e `sklearn_interface.py` do AutoGluon.


## Hiperparâmetros que ainda vão chegar

A pesquisa destes hiperparâmetros está nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. Contexto do v2: `hf_model` em `autogluon/mitra-classifier-2`, `fine_tune` falso, um estimador, espelho de colunas no default, sobre as dobras com oversampling.
2. `n_estimators`, subindo a partir de 1. O 32 do artigo só entra se os valores menores tiverem vencido.
3. `shuffle_features`.
4. `shuffle_classes`.
5. `use_random_transforms`.
6. `random_mirror_x` desligado, contra o default ligado.
7. Portão de fine-tuning: `fine_tune` verdadeiro e mais nada. Só fica se vencer o contexto já aceito.
8. `warmup_steps` em 10, o valor da receita v2.
9. `lr` em 3e-6, a taxa da receita v2 para binário até 16.384 linhas.
10. `patience`, só se o early stopping estiver cortando os 50 passos.
11. Mitra v1, só se o v2 não puder ser executado. Nesse caso o roteiro recomeça do bloco 0 com o checkpoint v1.
12. Estratégias de dados compartilhadas.
13. Bloco final no teste.
